# Nebulith Dynamic LLM Inference Server on Google Colab
このノートブックは、Google Colabの無料/有料GPU（T4 / A100）上で **OpenAI互換の高速推論サーバー（vLLM）** を起動し、**Cloudflare Tunnel** を通じてローカルの Nebulith 開発環境（`tools/llm_agent_dialogue.py`）から直接呼び出せるようにする環境です。

### 特徴:
- **APIキー必須**: 起動時にキーを生成し、vLLM を `--api-key` 付きで待ち受けます。キーはセルの出力に表示されるので、ローカルの `NEBULITH_LLM_API_KEY` に設定してください。
- **公開URLの既定値なし**: トンネルURLは実行のたびに変わります。リポジトリ側のスクリプトは `--endpoint` か `NEBULITH_LLM_ENDPOINT` が無いと接続しません。
- **推奨モデル**: `Qwen/Qwen2.5-7B-Instruct`（日本語・英語・未知構文の追従性能が極めて高く、NebulithのVSO語順・格標識・6段階音韻規則を正確に反映します）。
- **OpenAI互換**: ローカルPC側は `--provider colab --endpoint https://xxxx.trycloudflare.com/v1 --api-key <表示されたキー>` で接続します。`OPENAI_API_KEY` はこのトンネルへは送りません。


## Step 1: GPU確認とライブラリのセットアップ
メニューの「ランタイム」➔「ランタイムのタイプを変更」で **GPU（T4 または A100）** が選択されていることを確認して実行してください。

In [ ]:
# GPU情報の確認
!nvidia-smi

# vLLM と Cloudflare Tunnel (cloudflared) のインストール
!pip install -q "vllm>=0.6.0"
!curl -s -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("Setup completed successfully!")

## Step 2: vLLM 推論サーバーのバックグラウンド起動
OpenAI互換 API サーバーをポート 8000 で起動します。APIキーを生成して `--api-key` に渡し、認証のない公開待ち受けにはしません。


In [ ]:
import secrets
import subprocess
import time
import sys

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
PORT = 8000
API_KEY = secrets.token_urlsafe(32)

cmd = [
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--port", str(PORT),
    "--api-key", API_KEY,
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.90",
    "--dtype", "auto"
]

print("NEBULITH_LLM_API_KEY=" + API_KEY)
print("このキーをローカル環境変数 NEBULITH_LLM_API_KEY に設定してください。OPENAI_API_KEY は使いません。")
print(f"Starting vLLM server with model '{MODEL_ID}'...")
vllm_process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# サーバー起動待ち
for _ in range(120):
    line = vllm_process.stdout.readline()
    if line:
        sys.stdout.write(line)
        sys.stdout.flush()
    if "Uvicorn running on" in line or "Application startup complete" in line:
        print("\n[SUCCESS] vLLM Inference Server is now RUNNING on port 8000!")
        break
    time.sleep(0.5)


## Step 3: Cloudflare Tunnel の起動と接続URLの発行
ローカルPCからアクセスできるトンネルURLを発行します。URLと、Step 2 で表示した API キーの両方が必要です。


In [ ]:
import re
import subprocess

tunnel_cmd = ["cloudflared", "tunnel", "--url", "http://localhost:8000"]
tunnel_process = subprocess.Popen(tunnel_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

public_endpoint = None
for _ in range(40):
    line = tunnel_process.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_endpoint = match.group(0)
        break

if public_endpoint:
    v1_url = f"{public_endpoint}/v1"
    print("=" * 75)
    print("★ Nebulith Colab 推論サーバーが接続準備完了しました！")
    print(f"API Base Endpoint: {v1_url}")
    print(f"NEBULITH_LLM_API_KEY={API_KEY}")
    print("=" * 75)
    print("\n【ローカルPCのターミナルで実行するコマンド】")
    print("# 環境変数（OPENAI_API_KEY は api.openai.com 以外へ送らない）")
    print(f'export NEBULITH_LLM_ENDPOINT="{v1_url}"')
    print(f'export NEBULITH_LLM_API_KEY="{API_KEY}"')
    print()
    print("# 1. 接続テスト (Health Check)")
    print(f'python tools/llm_agent_dialogue.py --provider colab --endpoint "{v1_url}" --api-key "{API_KEY}" --health-check\n')
    print("# 2. 任意のトピックでAI対話創発 & 6段階審査を実行")
    print(f'python tools/llm_agent_dialogue.py --provider colab --endpoint "{v1_url}" --api-key "{API_KEY}" --topic "天体航路と星間通信"\n')
    print("# 3. 未対応フィードバックキューを自動トリアージ")
    print(f'python tools/triage_feedback_queue.py data/your_feedback.json --provider colab --endpoint "{v1_url}" --api-key "{API_KEY}"')
else:
    print("[WARN] Could not find tunnel URL. Check tunnel logs.")
    print(f"NEBULITH_LLM_API_KEY={API_KEY}")
